# Notebook 4 — CMF Unlearning: cmf_static (4a) + Post-Hoc Full Fine-Tune (4b)

**Paper:** *An Illusion of Unlearning?* (Gao et al., AISTATS 2026 · arXiv:2604.08271v1)

**Protocol — matches paper exactly (Table 1 / Appendix A.3):**
- Forget set = **one entire class** (all ~5 000 training images of that class).
- Retain set = all training images from the remaining 9 classes.
- Loop over all 10 CIFAR-10 classes as the forget class; results averaged (mean ± std).
- **Output/Probe/NCC** evaluated on the **held-out test set** (paper Appendix A.2).

**4a. cmf_static (Algorithm 2):** per-epoch `recompute_cmf → freeze W → update encoder`.
**4b. cmf_static + post-hoc full fine-tune:** LOAD the 4a checkpoint, fine-tune the
full model (encoder + W together) for k_posthoc epochs. No recompute_cmf during phase 2.

**Outputs:** `{method}_cmf_static_class{c}_seed{s}.pt` per run; `results_4a_cmf_static.csv` summary.

In [ ]:
import subprocess, sys
def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout: print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-2000:])
    return r.returncode
sh('pip install -q timm einops scikit-learn matplotlib seaborn pytorch-lightning torchmetrics')

In [ ]:
import os, sys, json, random, math, time, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib; import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})
print('PyTorch:', torch.__version__, '  CUDA:', torch.cuda.is_available())

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'
if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} remote set-url origin https://github.com/tiensinh2/CMF_Unlearning.git')
    sh(f'git -C {REPO_DIR} pull origin main')
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
result = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'],
                        capture_output=True, text=True)
REPO_COMMIT = result.stdout.strip() or 'main'
print('Repo commit:', REPO_COMMIT)

In [ ]:
CKPT_DATASET_DIR = '/kaggle/input/datasets/btk23021592/cmf-notebook1'
_CONFIG_CANDIDATES = [
    f'{CKPT_DATASET_DIR}/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/checkpoints/cmf_benchmark/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/cmf_benchmark/cmf_benchmark_config.json',
]
config_path = CKPT_ROOT_NB1 = None
for _p in _CONFIG_CANDIDATES:
    if os.path.exists(_p):
        config_path = _p; CKPT_ROOT_NB1 = os.path.dirname(_p); break
assert config_path, f'Cannot find cmf_benchmark_config.json'
with open(config_path) as f: NB1_CFG = json.load(f)

DATASET     = NB1_CFG['dataset']       # 'cifar10'
ARCH        = NB1_CFG['arch']          # 'resnet18'
NUM_CLASSES = NB1_CFG['num_classes']   # 10
TEST_MODE   = NB1_CFG.get('test_mode', False)

# Paper protocol: sweep all 10 classes as forget class; one seed.
FORGET_CLASSES = list(range(NUM_CLASSES))   # [0, 1, ..., 9]
SEEDS          = [0]
# theta_o: single fully-trained model from NB1.
THETA_O_SEED   = 0

# CMF experiment matrix
BASE_METHODS   = ['scrub', 'grad_ascent_descent', 'random_label', 'salun', 'tarun']
MEAN_SOURCES   = ['train']   # paper-faithful: class means from full training set
# Table 4: RL+CMF=4ep, SalUn+CMF=4ep, NegGrad++CMF=3ep, SCRUB+CMF=3ep, UNSIR+CMF=3ep
CMF_EPOCHS_BY_METHOD = {
    'random_label':        2 if TEST_MODE else 4,
    'salun':               2 if TEST_MODE else 4,
    'grad_ascent_descent': 1 if TEST_MODE else 3,
    'scrub':               1 if TEST_MODE else 3,
    'tarun':               1 if TEST_MODE else 3,
}
K_POSTHOC   = [2, 5, 10]
PHASE2_DATA = ['retain_only', 'retain_plus_forget']   # paper_hparams.py POSTHOC_PHASE2_DATA
HPARAM_SOURCE = 'table4'

CKPT_ROOT = '/kaggle/working/checkpoints/cmf_paper'
os.makedirs(f'{CKPT_ROOT}/cmf_static',  exist_ok=True)
os.makedirs(f'{CKPT_ROOT}/cmf_posthoc', exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'CMF_EPOCHS_BY_METHOD={CMF_EPOCHS_BY_METHOD}  device={device}')

In [ ]:
import torchvision, torchvision.transforms as transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Training set — augmented (for unlearn methods)
full_train      = torchvision.datasets.CIFAR10('/kaggle/working/data', train=True,
                                               download=True,  transform=transform_train)
# Training set — eval transform (for probe/NCC feature extraction, paper §3.2 / eq.3)
full_train_eval = torchvision.datasets.CIFAR10('/kaggle/working/data', train=True,
                                               download=False, transform=transform_test)
# Test set — for ALL final metric evaluation (paper Appendix A.2)
test_set = torchvision.datasets.CIFAR10('/kaggle/working/data', train=False,
                                        download=True, transform=transform_test)

# Pre-index test set by class label
test_targets = torch.tensor(test_set.targets)   # [10000]
TEST_CLASS_IDX = {
    c: (test_targets == c).nonzero(as_tuple=True)[0].tolist()
    for c in range(NUM_CLASSES)
}

# Pre-index training set by class label
train_targets = torch.tensor(full_train.targets)  # [50000]
TRAIN_CLASS_IDX = {
    c: (train_targets == c).nonzero(as_tuple=True)[0].tolist()
    for c in range(NUM_CLASSES)
}

print(f'Train: {len(full_train)}  Test: {len(test_set)}')
print(f'Test samples per class: {len(TEST_CLASS_IDX[0])} (expected 1000)')

In [ ]:
import argparse
from unlearn.cmf_weights import ModelModule
from unlearn.cmf_two_stage import CMFWeightsTrainable
from unlearn import unlear_func

# LRs for CMF variants (Table 4, PDF lines 2390–2434)
CMF_LR = {
    'scrub':               5e-3,
    'grad_ascent_descent': 1e-4,
    'random_label':        2e-3,
    'salun':               2e-3,
    'tarun':               5e-5,
}
CMF_BATCH = {'scrub': 64}
CMF_BATCH_DEFAULT = 128


def make_cmf_args(base_method, lr, epochs, mean_source, forget_class, forget_train_idx,
                  retain_train_idx, seed=0):
    """Build args namespace for one whole-class CMF unlearning run."""
    return argparse.Namespace(
        dataset=DATASET, arch=ARCH, num_classes=NUM_CLASSES,
        class_label_names=list(range(NUM_CLASSES)),
        unlearn_method=f'{base_method}_CMF_RemoveFC',
        unlearn_class=[forget_class],   # single class — matches paper
        batch_size=128, test_batch_size=256,
        lr=lr, momentum=0.9, weight_decay=5e-4,
        epochs_or_steps=epochs,
        seed=seed,
        num_retain_samples=len(retain_train_idx),
        num_forget_samples=len(forget_train_idx),
        grad_norm_clip=1.0,
        SVD_alpha_r=1000, SVD_alpha_f=30, SVD_samples=900, SVD_max_patches=10000,
        freeze_except_last=False,
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=epochs,
        salun_threshold=0.5,
        tarun_impair_lr=lr, tarun_samples_per_class=1000,
        dry_run=TEST_MODE, no_cuda=False, no_mps=True, gamma=0.5,
        data_path='/kaggle/working/data',
        remove_FC=True, CMFClassifier=True, CMF_momentum=0.9,
        pretrained=False, temperature=1.0,
        prob_batch_size=256, lp_every=0,
        mean_source=mean_source,
        repo_commit=REPO_COMMIT, test_mode=TEST_MODE,
    )


def build_cmf_model(args):
    return ModelModule(args).to(device)


@torch.no_grad()
def eval_acc(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        correct += (model(x.to(device)).argmax(1).cpu() == y).sum().item()
        total   += y.size(0)
    return 100.0 * correct / max(total, 1)


@torch.no_grad()
def cmf_extract_features(model, loader):
    """Extract CMF-preprocessed features: ẑ = normalize(normalize(f) − μ)."""
    model.eval()
    feats, labs = [], []
    for x, y in loader:
        x = x.to(device)
        f = model.extract_features(x)
        z = model._preprocess_feats_for_cmf(f)
        feats.append(z.cpu()); labs.append(y)
    return torch.cat(feats), torch.cat(labs)


def run_probe_cmf(model, train_retain_ldr, train_forget_ldr,
                  test_retain_ldr, test_forget_ldr, n_epochs=50):
    """
    Paper §3.2: probe trained on ALL training features D_r ∪ D_f,
    evaluated on test-set retain and test-set forget subsets.
    """
    Xtr, ytr = cmf_extract_features(model, train_retain_ldr)
    Xfg, yfg = cmf_extract_features(model, train_forget_ldr)
    Xall = torch.cat([Xtr, Xfg])
    yall = torch.cat([ytr, yfg])

    head = nn.Linear(Xall.size(1), NUM_CLASSES).to(device)
    opt  = optim.SGD(head.parameters(), lr=1e-2, momentum=0.9)
    ldr  = torch.utils.data.DataLoader(
               torch.utils.data.TensorDataset(Xall, yall), batch_size=256, shuffle=True)
    for _ in range(n_epochs):
        head.train()
        for bx, by in ldr:
            opt.zero_grad()
            F.cross_entropy(head(bx.to(device)), by.to(device)).backward()
            opt.step()
    head.eval()

    with torch.no_grad():
        Xte_r, yte_r = cmf_extract_features(model, test_retain_ldr)
        Xte_f, yte_f = cmf_extract_features(model, test_forget_ldr)
        ret_acc = (head(Xte_r.to(device)).argmax(1).cpu() == yte_r).float().mean().item() * 100
        fgt_acc = (head(Xte_f.to(device)).argmax(1).cpu() == yte_f).float().mean().item() * 100
    return ret_acc, fgt_acc


def run_ncc_cmf(model, train_retain_ldr, train_forget_ldr,
                test_retain_ldr, test_forget_ldr):
    """
    Paper eq. 3: class means from ALL training samples D_r ∪ D_f (CMF features).
    Evaluated on test-set retain and test-set forget subsets.
    """
    Xtr, ytr = cmf_extract_features(model, train_retain_ldr)
    Xfg, yfg = cmf_extract_features(model, train_forget_ldr)
    Xall = torch.cat([Xtr, Xfg])
    yall = torch.cat([ytr, yfg])
    means = []
    for c in range(NUM_CLASSES):
        mask = (yall == c)
        mu = Xall[mask].mean(0) if mask.any() else torch.zeros(Xall.size(1))
        means.append(mu)
    M = torch.stack(means)  # [C, D]

    Xte_r, yte_r = cmf_extract_features(model, test_retain_ldr)
    Xte_f, yte_f = cmf_extract_features(model, test_forget_ldr)
    ret_pred = torch.cdist(Xte_r.unsqueeze(0), M.unsqueeze(0)).squeeze(0).argmin(dim=1)
    fgt_pred = torch.cdist(Xte_f.unsqueeze(0), M.unsqueeze(0)).squeeze(0).argmin(dim=1)
    ret_acc  = (ret_pred == yte_r).float().mean().item() * 100
    fgt_acc  = (fgt_pred == yte_f).float().mean().item() * 100
    return ret_acc, fgt_acc


def eval_cmf_three_metrics(model,
                           test_retain_ldr, test_forget_ldr,
                           train_retain_eval_ldr, train_forget_eval_ldr):
    """
    Three-metric evaluation matching paper Table 1.

    Output  — direct model accuracy on HELD-OUT TEST-SET retain / forget images.
    Probe   — linear head on ALL TRAINING CMF features (D_r ∪ D_f), eval on test set.
    NCC     — class means from ALL TRAINING CMF features, eval on test set.
    """
    out_ret = eval_acc(model, test_retain_ldr)
    out_fgt = eval_acc(model, test_forget_ldr)

    lp_ret, lp_fgt = run_probe_cmf(
        model,
        train_retain_eval_ldr, train_forget_eval_ldr,
        test_retain_ldr,       test_forget_ldr
    )

    ncc_ret, ncc_fgt = run_ncc_cmf(
        model,
        train_retain_eval_ldr, train_forget_eval_ldr,
        test_retain_ldr,       test_forget_ldr
    )

    return {
        'output_retain_acc': out_ret, 'output_forget_acc': out_fgt,
        'probe_retain_acc':  lp_ret,  'probe_forget_acc':  lp_fgt,
        'ncc_retain_acc':    ncc_ret, 'ncc_forget_acc':    ncc_fgt,
    }


print('CMF helpers ready.')

In [ ]:
# ─── 4a: cmf_static (Algorithm 2) ────────────────────────────────────────────
# Paper protocol: forget one entire class at a time, sweep all 10 classes.
# Output/Probe/NCC evaluated on the held-out test set.

# Load theta_o once (single model used as starting point for all runs)
theta_o_path = f'{CKPT_ROOT_NB1}/pre_train/theta_o_seed{THETA_O_SEED}.pt'
if TEST_MODE: theta_o_path = theta_o_path.replace('.pt', '_testmode.pt')
assert os.path.exists(theta_o_path), f'Missing theta_o: {theta_o_path}'
ck_o = torch.load(theta_o_path, map_location=device)
theta_o_state = ck_o.get('model_state_dict', ck_o)

results_4a = []

for forget_class in FORGET_CLASSES:
    for seed in SEEDS:

        # ── Build whole-class forget / retain splits ──────────────────────────
        forget_train_idx = TRAIN_CLASS_IDX[forget_class]           # ~5 000
        retain_train_idx = [
            i for c in range(NUM_CLASSES)
            if c != forget_class
            for i in TRAIN_CLASS_IDX[c]
        ]                                                            # ~45 000

        # Training-set loaders (augmented) — for unlearn methods
        retain_loader = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train, retain_train_idx),
            batch_size=128, shuffle=True, num_workers=2)
        forget_loader = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train, forget_train_idx),
            batch_size=128, shuffle=True, num_workers=2)
        full_train_loader = torch.utils.data.DataLoader(
            full_train, batch_size=256, shuffle=False, num_workers=2)

        # Training-set loaders (eval transform) — for probe/NCC feature extraction
        train_retain_eval_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train_eval, retain_train_idx),
            batch_size=256, shuffle=False, num_workers=2)
        train_forget_eval_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train_eval, forget_train_idx),
            batch_size=256, shuffle=False, num_workers=2)

        # ── Test-set loaders ─────────────────────────────────────────────────
        test_forget_idx = TEST_CLASS_IDX[forget_class]             # 1 000
        test_retain_idx = [
            i for c in range(NUM_CLASSES)
            if c != forget_class
            for i in TEST_CLASS_IDX[c]
        ]                                                            # 9 000
        test_forget_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(test_set, test_forget_idx),
            batch_size=256, shuffle=False, num_workers=2)
        test_retain_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(test_set, test_retain_idx),
            batch_size=256, shuffle=False, num_workers=2)

        for base_method in BASE_METHODS:
            for mean_source in MEAN_SOURCES:
                tag = f'{base_method}_cmf_static_{mean_source}_class{forget_class}_seed{seed}'
                if TEST_MODE: tag += '_testmode'
                ckpt_path = f'{CKPT_ROOT}/cmf_static/{tag}.pt'

                if os.path.exists(ckpt_path):
                    print(f'[4a {tag}] exists — skipping.')
                    ck = torch.load(ckpt_path, map_location=device)
                    results_4a.append(ck['metrics'])
                    continue

                cmf_epochs = CMF_EPOCHS_BY_METHOD.get(base_method, 3)
                lr    = CMF_LR.get(base_method, 1e-3)
                batch = CMF_BATCH.get(base_method, CMF_BATCH_DEFAULT)
                print(f'\n[4a {tag}] cmf_epochs={cmf_epochs}')
                torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

                args = make_cmf_args(base_method, lr, cmf_epochs, mean_source,
                                     forget_class, forget_train_idx, retain_train_idx,
                                     seed=seed)
                args.batch_size = batch
                if base_method == 'scrub':
                    args.scrub_del_bsz  = 64
                    args.scrub_sgda_bsz = 64

                model = build_cmf_model(args)
                model.encoder.load_state_dict(theta_o_state, strict=False)

                mean_loader = full_train_loader if mean_source == 'train' else retain_loader
                model.eval()
                model.recompute_cmf(mean_loader, device=device)

                # Internal test_loader for unlearn methods: full test set
                _test_loader_full = torch.utils.data.DataLoader(
                    test_set, batch_size=256, shuffle=False, num_workers=2)

                dispatch_key = f'{base_method}_CMF_RemoveFC'
                fn = unlear_func[dispatch_key]

                t0 = time.time()
                try:
                    model = fn(
                        args=args, model=model, device=device,
                        retain_loader=retain_loader, forget_loader=forget_loader,
                        train_loader=mean_loader, test_loader=_test_loader_full,
                        optimizer=None, epochs=cmf_epochs,
                        test_forget_loader=test_forget_ldr,
                        train_dataset=full_train,
                        val_index=retain_train_idx,
                    )
                except Exception as e:
                    import traceback; traceback.print_exc()
                    print(f'  ERROR: {e}'); continue
                wall_min = (time.time() - t0) / 60

                # Final recompute_cmf
                model.eval()
                model.recompute_cmf(mean_loader, device=device)

                # ── Evaluation on test set ────────────────────────────────────
                metrics = eval_cmf_three_metrics(
                    model,
                    test_retain_ldr,       # test-set retain
                    test_forget_ldr,       # test-set forget
                    train_retain_eval_ldr, # training-set retain (probe/NCC features)
                    train_forget_eval_ldr  # training-set forget (probe/NCC features)
                )
                metrics.update({
                    'method': base_method, 'mean_source': mean_source,
                    'forget_class': forget_class, 'seed': seed, 'stage': '4a',
                    'cmf_epochs': cmf_epochs, 'lr': lr, 'batch': batch,
                    'wall_clock_minutes': wall_min,
                    'n_forget_train': len(forget_train_idx),
                    'n_retain_train': len(retain_train_idx),
                    'protocol': 'whole_class_single',
                    'hparam_source': HPARAM_SOURCE,
                })

                torch.save({
                    'model_state_dict': model.state_dict(),
                    'config': {
                        'base_method': base_method, 'mean_source': mean_source,
                        'dataset': DATASET, 'arch': ARCH, 'num_classes': NUM_CLASSES,
                        'forget_class': forget_class, 'seed': seed,
                        'cmf_epochs': cmf_epochs, 'lr': lr, 'batch': batch,
                        'protocol': 'whole_class_single',
                        'hparam_source': HPARAM_SOURCE,
                        'repo_commit': REPO_COMMIT, 'test_mode': TEST_MODE,
                    },
                    'seed': seed, 'metrics': metrics,
                }, ckpt_path)
                print(f'  Saved {ckpt_path}')
                print(f'  out  R={metrics["output_retain_acc"]:.2f}%  F={metrics["output_forget_acc"]:.2f}%')
                print(f'  prob R={metrics["probe_retain_acc"]:.2f}%  F={metrics["probe_forget_acc"]:.2f}%')
                results_4a.append(metrics)


# ── Aggregate 4a results ────────────────────────────────────────────────────
df_4a = pd.DataFrame(results_4a)
df_4a.to_csv(f'{CKPT_ROOT}/results_4a_cmf_static.csv', index=False)
print('\n=== 4a cmf_static results saved ===')
if not df_4a.empty:
    metric_cols = ['output_retain_acc','output_forget_acc',
                   'probe_retain_acc','probe_forget_acc','ncc_retain_acc','ncc_forget_acc']
    summary = (
        df_4a.groupby(['method', 'mean_source'])[metric_cols]
        .agg(['mean', 'std'])
        .round(2)
    )
    summary.columns = ['_'.join(c) for c in summary.columns]
    summary.to_csv(f'{CKPT_ROOT}/results_4a_cmf_static_summary.csv')
    print('\n=== Mean across all forget classes ===')
    print(summary[[c for c in summary.columns if c.endswith('_mean')]].to_string())

In [ ]:
# ─── 4b: post-hoc W calibration ──────────────────────────────────────────────
# LOAD 4a checkpoint; fine-tune full model (encoder + W) with recompute_cmf each epoch.

results_4b = []

for forget_class in FORGET_CLASSES:
    for seed in SEEDS:

        # ── Rebuild splits and loaders (same as 4a) ──────────────────────────
        forget_train_idx = TRAIN_CLASS_IDX[forget_class]
        retain_train_idx = [
            i for c in range(NUM_CLASSES)
            if c != forget_class
            for i in TRAIN_CLASS_IDX[c]
        ]
        retain_loader = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train, retain_train_idx),
            batch_size=128, shuffle=True, num_workers=2)
        forget_loader = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train, forget_train_idx),
            batch_size=128, shuffle=True, num_workers=2)
        train_retain_eval_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train_eval, retain_train_idx),
            batch_size=256, shuffle=False, num_workers=2)
        train_forget_eval_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train_eval, forget_train_idx),
            batch_size=256, shuffle=False, num_workers=2)

        test_forget_idx = TEST_CLASS_IDX[forget_class]
        test_retain_idx = [
            i for c in range(NUM_CLASSES)
            if c != forget_class
            for i in TEST_CLASS_IDX[c]
        ]
        test_forget_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(test_set, test_forget_idx),
            batch_size=256, shuffle=False, num_workers=2)
        test_retain_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(test_set, test_retain_idx),
            batch_size=256, shuffle=False, num_workers=2)

        for base_method in BASE_METHODS:
            for mean_source in MEAN_SOURCES:
                # Load 4a checkpoint
                tag_4a = (f'{base_method}_cmf_static_{mean_source}_'
                          f'class{forget_class}_seed{seed}')
                if TEST_MODE: tag_4a += '_testmode'
                ckpt_4a = f'{CKPT_ROOT}/cmf_static/{tag_4a}.pt'
                if not os.path.exists(ckpt_4a):
                    print(f'[4b] Missing 4a ckpt: {ckpt_4a} — skipping.')
                    continue

                ck_4a = torch.load(ckpt_4a, map_location=device)
                metrics_4a = ck_4a['metrics']

                retain_set_sub = torch.utils.data.Subset(full_train, retain_train_idx)
                forget_set_sub = torch.utils.data.Subset(full_train, forget_train_idx)
                mean_loader = (torch.utils.data.DataLoader(
                    full_train, batch_size=256, shuffle=False, num_workers=2)
                    if mean_source == 'train' else retain_loader)

                for k_posthoc in K_POSTHOC:
                    for phase2_data in PHASE2_DATA:
                        tag = (f'{base_method}_cmf_static_posthoc_k{k_posthoc}_'
                               f'{phase2_data}_{mean_source}_class{forget_class}_seed{seed}')
                        if TEST_MODE: tag += '_testmode'
                        ckpt_path = f'{CKPT_ROOT}/cmf_posthoc/{tag}.pt'

                        if os.path.exists(ckpt_path):
                            print(f'[4b {tag}] exists — skipping.')
                            ck = torch.load(ckpt_path, map_location=device)
                            results_4b.append(ck['metrics'])
                            continue

                        print(f'\n[4b {tag}]')
                        lr   = CMF_LR.get(base_method, 1e-3)
                        batch = CMF_BATCH.get(base_method, CMF_BATCH_DEFAULT)
                        cmf_epochs = CMF_EPOCHS_BY_METHOD.get(base_method, 3)
                        args = make_cmf_args(base_method, lr, cmf_epochs, mean_source,
                                             forget_class, forget_train_idx, retain_train_idx)
                        args.batch_size = batch
                        if base_method == 'scrub':
                            args.scrub_del_bsz  = 64
                            args.scrub_sgda_bsz = 64

                        # Load 4a model
                        model = build_cmf_model(args)
                        model.load_state_dict(ck_4a['model_state_dict'], strict=False)
                        model = model.to(device)

                        # Phase 2: full model fine-tuning (encoder + W together)
                        for p in model.parameters():
                            p.requires_grad_(True)

                        # Stage-2 data loader
                        if phase2_data == 'retain_plus_forget':
                            from torch.utils.data import ConcatDataset
                            s2_ds  = ConcatDataset([retain_set_sub, forget_set_sub])
                            s2_ldr = torch.utils.data.DataLoader(
                                s2_ds, batch_size=128, shuffle=True)
                        else:
                            s2_ldr = retain_loader

                        opt_s2 = optim.SGD(model.parameters(),
                                           lr=CMF_LR.get(base_method, 1e-3),
                                           momentum=0.9, weight_decay=5e-4,
                                           nesterov=True)

                        s2_log = []
                        for ep in range(1, k_posthoc + 1):
                            model.train()
                            ep_loss = n_batches = 0
                            for xb, yb in s2_ldr:
                                xb, yb = xb.to(device), yb.to(device)
                                opt_s2.zero_grad()
                                loss, _ = model.forward_a((xb, yb), stage='train')
                                loss.backward()
                                opt_s2.step()
                                ep_loss  += loss.item()
                                n_batches += 1
                                if TEST_MODE: break

                            model.eval()
                            ep_metrics = eval_cmf_three_metrics(
                                model, test_retain_ldr, test_forget_ldr,
                                train_retain_eval_ldr, train_forget_eval_ldr
                            )


                            print(f'  [4b ep{ep}] out R={ep_metrics["output_retain_acc"]:.2f}%'
                                  f' F={ep_metrics["output_forget_acc"]:.2f}%')
                            s2_log.append({'epoch': ep, **ep_metrics})

                        # Final metrics
                        final_metrics = eval_cmf_three_metrics(
                            model, test_retain_ldr, test_forget_ldr,
                            train_retain_eval_ldr, train_forget_eval_ldr
                        )
                        final_metrics.update({
                            'method': base_method, 'mean_source': mean_source,
                            'stage': '4b', 'k_posthoc': k_posthoc,
                            'phase2_data': phase2_data,
                            'forget_class': forget_class, 'seed': seed,
                            'protocol': 'whole_class_single',
                        })

                        torch.save({
                            'model_state_dict': model.state_dict(),
                            'config': {
                                'base_method': base_method, 'mean_source': mean_source,
                                'k_posthoc': k_posthoc, 'phase2_data': phase2_data,
                                'dataset': DATASET, 'arch': ARCH, 'num_classes': NUM_CLASSES,
                                'forget_class': forget_class, 'seed': seed,
                                'protocol': 'whole_class_single',
                                'source_4a_ckpt': ckpt_4a,
                                'repo_commit': REPO_COMMIT, 'test_mode': TEST_MODE,
                            },
                            'seed': seed, 'metrics': final_metrics,
                            'stage2_log': s2_log,
                            'metrics_4a_probe_retain': metrics_4a.get('probe_retain_acc'),
                            'metrics_4a_ncc_retain':   metrics_4a.get('ncc_retain_acc'),
                        }, ckpt_path)
                        print(f'  Saved {ckpt_path}')
                        results_4b.append(final_metrics)

df_4b = pd.DataFrame(results_4b)
df_4b.to_csv(f'{CKPT_ROOT}/results_4b_cmf_posthoc.csv', index=False)
print('\n=== 4b post-hoc W results saved ===')

In [ ]:
# ─── Summary table: 4a vs 4b (mean ± std over all 10 forget classes) ─────────
metric_cols = ['output_retain_acc','output_forget_acc',
               'probe_retain_acc','probe_forget_acc','ncc_retain_acc','ncc_forget_acc']

for base_method in BASE_METHODS:
    for mean_source in MEAN_SOURCES:
        sub4a = df_4a[(df_4a['method'] == base_method) &
                      (df_4a['mean_source'] == mean_source)] if not df_4a.empty else pd.DataFrame()
        sub4b = df_4b[(df_4b['method'] == base_method) &
                      (df_4b['mean_source'] == mean_source)] if not df_4b.empty else pd.DataFrame()
        if sub4a.empty and sub4b.empty: continue

        print(f'\n=== {base_method.upper()} | mean_source={mean_source} ===')

        def fmt_row(df, label):
            row = [label]
            for c in metric_cols:
                if c in df.columns:
                    v = df[c].dropna()
                    row.append(f'{v.mean():.1f}±{v.std():.1f}' if len(v) > 0 else 'N/A')
                else:
                    row.append('N/A')
            return row

        header = ['Variant'] + metric_cols
        rows = [fmt_row(sub4a, 'cmf_static(4a)')]
        for k in K_POSTHOC:
            for ph in PHASE2_DATA:
                s = sub4b
                if 'k_posthoc' in s.columns:   s = s[s['k_posthoc'] == k]
                if 'phase2_data' in s.columns: s = s[s['phase2_data'] == ph]
                if not s.empty:
                    rows.append(fmt_row(s, f'posthoc_k{k}_{ph}'))

        print(pd.DataFrame(rows, columns=header).to_string(index=False))